# Data Cleaning — Left-Wing Non-Fiction
Loads `Data/Raw/non fiction/leftpolitics_raw.csv`, cleans it, and saves to `Data/non fiction/leftpolitics_clean.csv`.

In [1]:
import pandas as pd
import os

In [2]:
RAW_PATH = "Data/Raw/non fiction/leftpolitics_raw.csv"
CLEAN_DIR = "Data/non fiction"
CLEAN_PATH = os.path.join(CLEAN_DIR, "leftpolitics_clean.csv")

df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()

Loaded 1311 rows, 7 columns


,title,author,year_published,subjects,edition_count,open_library_key,queried_author
0,Le PROBLÈME NATIONAL KAMERUNAIS,Achille Mbembe,1984.0,Politics and government | Nationalism | Nation...,1,/works/OL31757685W,Achille Mbembe
1,Les jeunes et l'ordre politique en Afrique noire,Achille Mbembe,1985.0,Youth | Political activity,1,/works/OL2537390W,Achille Mbembe
2,Les JEUNES ET L'ORDRE POLITIQUE EN AFRIQUE NOIRE,Achille Mbembe,1985.0,Youth | Political activity,1,/works/OL31741886W,Achille Mbembe
3,Afriques indociles,Achille Mbembe,1988.0,Afrique | Religion et État | Christianisme | C...,1,/works/OL9171632W,Achille Mbembe
4,"La naissance du maquis dans le Sud-Cameroun, 1...",Achille Mbembe,1996.0,History | Insurgency | Politics and government...,2,/works/OL2537389W,Achille Mbembe


In [3]:
print("=== Before cleaning ===")
print(df.dtypes)
print()
print(df.isnull().sum())

=== Before cleaning ===
title                object
author               object
year_published      float64
subjects             object
edition_count         int64
open_library_key     object
queried_author       object
dtype: object

title                 0
author                0
year_published       14
subjects            343
edition_count         0
open_library_key      0
queried_author        0
dtype: int64


In [4]:
df_clean = df.copy()

# --- Strip whitespace from all string columns ---
str_cols = df_clean.select_dtypes(include="object").columns
df_clean[str_cols] = df_clean[str_cols].apply(lambda col: col.str.strip())

# --- Drop rows missing title or author ---
before = len(df_clean)
df_clean = df_clean.dropna(subset=["title", "author"])
df_clean = df_clean[df_clean["title"].str.len() > 0]
df_clean = df_clean[df_clean["author"].str.len() > 0]
print(f"Dropped {before - len(df_clean)} rows missing title or author")

# --- Standardize author names: title-case, collapse extra spaces ---
def standardize_name(name):
    return " ".join(name.title().split())

df_clean["author"] = df_clean["author"].apply(standardize_name)
df_clean["queried_author"] = df_clean["queried_author"].apply(standardize_name)

# --- Make year numeric ---
df_clean["year_published"] = pd.to_numeric(df_clean["year_published"], errors="coerce")

# --- Drop exact duplicates (title + author) ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title", "author"])
print(f"Dropped {before - len(df_clean)} duplicate rows")

# --- Reset index ---
df_clean = df_clean.reset_index(drop=True)

print(f"\nClean dataset: {len(df_clean)} rows")
df_clean.head(10)

Dropped 0 rows missing title or author
Dropped 3 duplicate rows

Clean dataset: 1308 rows


,title,author,year_published,subjects,edition_count,open_library_key,queried_author
0,Le PROBLÈME NATIONAL KAMERUNAIS,Achille Mbembe,1984.0,Politics and government | Nationalism | Nation...,1,/works/OL31757685W,Achille Mbembe
1,Les jeunes et l'ordre politique en Afrique noire,Achille Mbembe,1985.0,Youth | Political activity,1,/works/OL2537390W,Achille Mbembe
2,Les JEUNES ET L'ORDRE POLITIQUE EN AFRIQUE NOIRE,Achille Mbembe,1985.0,Youth | Political activity,1,/works/OL31741886W,Achille Mbembe
3,Afriques indociles,Achille Mbembe,1988.0,Afrique | Religion et État | Christianisme | C...,1,/works/OL9171632W,Achille Mbembe
4,"La naissance du maquis dans le Sud-Cameroun, 1...",Achille Mbembe,1996.0,History | Insurgency | Politics and government...,2,/works/OL2537389W,Achille Mbembe
5,Du gouvernement privé indirect,Achille Mbembe,1999.0,Politics and government | Privatization,1,/works/OL2537388W,Achille Mbembe
6,De la postcolonie,Achille Mbembe,2000.0,Politics and government | Social conditions | ...,9,/works/OL2537387W,Achille Mbembe
7,On private indirect government,Achille Mbembe,2000.0,Politics and government | Privatization,1,/works/OL2537391W,Achille Mbembe
8,Globalization,"Arjun Appadurai, Achille Mbembe, Philippe Reka...",2001.0,Globalization | International relations,2,/works/OL25363313W,Achille Mbembe
9,On Private Indirect Government,Achille Mbembe,2002.0,"Africa, sub-saharan, politics and government |...",1,/works/OL9171634W,Achille Mbembe


In [5]:
print("=== After cleaning ===")
print(df_clean.dtypes)
print()
print(df_clean.isnull().sum())

=== After cleaning ===
title                object
author               object
year_published      float64
subjects             object
edition_count         int64
open_library_key     object
queried_author       object
dtype: object

title                 0
author                0
year_published       13
subjects            342
edition_count         0
open_library_key      0
queried_author        0
dtype: int64


In [6]:
os.makedirs(CLEAN_DIR, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(df_clean)} rows to {CLEAN_PATH}")

Saved 1308 rows to Data/non fiction/leftpolitics_clean.csv
